# {{TITLE}}

Describe the idea in one sentence here; the app shows it under the name.

Strategy `{{STRATEGY_ID}}`.

**How the app runs this notebook:** every cell, top to bottom (like Run All), with the strategies Python environment, from this notebook's folder. It sets `SIMPLI_STRATEGY_ID` and, for a dry run, `OMS_DRY_RUN=1`; `OMS()` reads both. Printed output becomes the run log, and an error in any cell marks the run failed.

**The only rule:** decide target weights in step 2, then step 3 calls `oms.rebalance()`.

In [ ]:
import os
from oms_client import OMS, latest_prices

STRATEGY_ID = os.environ.get("SIMPLI_STRATEGY_ID", "{{STRATEGY_ID}}")
MARKET_HOURS_ONLY = True   # outside market hours, print the plan instead of queueing orders
MIN_TRADE = 1.00           # skip changes under $1

## 1. Connect

In [ ]:
oms = OMS(STRATEGY_ID)
if MARKET_HOURS_ONLY and not oms.market_is_open():
    oms.plan_only("Market is closed")
assert oms.strategy() is not None, f"{STRATEGY_ID} is not registered in the OMS. Register it on the Strategies tab."
print(f"{STRATEGY_ID}: cash ${oms.strategy()['cash_remaining']:,.2f} | dry run: {oms.dry_run}")

## 2. Signal

Set `target` to `{symbol: fraction of this strategy's equity}`. Leftover weight stays in cash. Anything the strategy holds that isn't in `target` gets sold.

In [ ]:
target = {
    # "SPY": 0.5,
    # "QQQ": 0.5,
}
print("Target:", target or "all cash")

## 3. Trade

In [ ]:
prices = latest_prices(set(target) | set(oms.holdings()))
oms.rebalance(target, prices, min_trade=MIN_TRADE)